In [1]:
import pandas as pd
from tqdm import tqdm
from keybert import KeyBERT
import yake
import numpy as np
from sentence_transformers import SentenceTransformer


In [2]:
# Load Phase 1 outputs

df_train = pd.read_json("phase1_train_with_L4.json", lines=True)
df_validate = pd.read_json("phase1_validate_with_L4.json", lines=True)
df_test = pd.read_json("phase1_test_with_L4.json", lines=True)


In [3]:
df_train.head()

,Title,BrandInfo.BrandName,ProductName,Category.Name.Value,SummaryDescription.LongSummaryDescription,SummaryDescription.ShortSummaryDescription,Description.LongProductName,Description.LongDesc,pathlist_names,Level1,Level2,Level3,Level4,raw_text,cleaned_text
0,ASUS K31CD-IT049T PC 6th gen Intel® Core™ i7 i...,ASUS,K31CD-IT049T,PCs/Workstations,ASUS K31CD-IT049T. Processor frequency: 3.4 GH...,"ASUS K31CD-IT049T, 3.4 GHz, 6th gen Intel® Cor...","Intel Core i7-6700 (8M Cache, 3.4GHz), 16GB RA...",<b>Smart Multimedia Performance</b><br>\nVivoP...,Computers & Electronics>Computers>PCs/Workstat...,Computers & Electronics,Computers,PCs/Workstations,None,ASUS K31CD-IT049T PC 6th gen Intel® Core™ i7 i...,asus k31cd-it049t pc 6th gen i7 i7-6700 16 gb ...
1,HP 686915-A41 notebook spare part Keyboard,HP,686915-A41,Notebook Spare Parts,HP 686915-A41. Type: Keyboard. Keyboard langua...,"HP 686915-A41, Keyboard, Belgian, Keyboard bac...",Keyboard in midnight black finish with backlig...,,Computers & Electronics>Computers>Notebook Par...,Computers & Electronics,Computers,Notebook Parts & Accessories,Notebook Spare Parts,HP 686915-A41 notebook spare part Keyboard HP ...,hp 686915-a41 notebook spare part keyboard hp ...
2,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,C2G,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,Fibre Optic Cables,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,Get the performance you demand at a price that...,Computers & Electronics>Computer Cables>Fibre ...,Computers & Electronics,Computer Cables,Fibre Optic Cables,None,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,c2g 1m plenum-rated duplex single-mode fiber p...
3,HP FA889AA Battery,HP,FA889AA,Handheld Mobile Computer Spare Parts,"HP FA889AA. Product type: Battery, Product col...","HP FA889AA, Battery, White, Lithium-Ion (Li-Io...","1100 mAh, Lithium Ion, Standard Battery",Keeping an extra source of power nearby means ...,Computers & Electronics>Computers>Handheld Mob...,Computers & Electronics,Computers,Handheld Mobile Computer Spare Parts,None,HP FA889AA Battery HP FA889AA Handheld Mobile ...,hp fa889aa battery hp fa889aa handheld mobile ...
4,Lenovo ThinkStation C30 Intel® Xeon® E5 Family...,Lenovo,C30,PCs/Workstations,Lenovo ThinkStation C30. Processor frequency: ...,"Lenovo ThinkStation C30, 2 GHz, Intel® Xeon® E...","Intel Xeon E5-2620 (15M Cache, 2.00 GHz, 7.20 ...",The C30 builds on its award-winning design as ...,Computers & Electronics>Computers>PCs/Workstat...,Computers & Electronics,Computers,PCs/Workstations,None,Lenovo ThinkStation C30 Intel® Xeon® E5 Family...,lenovo thinkstation c30 e5 family e5-2620 4 gb...


In [4]:
# Merge into a single DataFrame so embeddings are computed only once
df_all = pd.concat([
    df_train.assign(dataset="train"),
    df_validate.assign(dataset="validate"),
    df_test.assign(dataset="test")
]).reset_index(drop=True)

print("Total rows for keyword extraction:", len(df_all))

Total rows for keyword extraction: 6000


In [5]:

#  Initialize keyword extractors and embedding model

kw_model = KeyBERT(model='all-MiniLM-L6-v2')      # KeyBERT with SBERT backbone
yake_extractor = yake.KeywordExtractor(top=10, n=1)  # YAKE for top 10 unigrams
model = SentenceTransformer("all-MiniLM-L6-v2")   # SBERT for embeddings


In [6]:

#Precompute embeddings for all documents
embeddings = model.encode(
    df_all["cleaned_text"].tolist(),
    show_progress_bar=True,
    batch_size=32
)
df_all["embedding"] = embeddings.tolist()


Batches:   0%|          | 0/188 [00:00<?, ?it/s]

In [7]:
# ---------------------------
# 4️⃣ Define keyword extraction functions
# ---------------------------
def extract_yake(text):
    try:
        return [kw for kw, score in yake_extractor.extract_keywords(text)]
    except:
        return []

def extract_keybert(text, emb):
    try:
        return [kw for kw, score in kw_model.extract_keywords(
            text,
            top_n=10,
            use_mmr=True,       # diversity
            diversity=0.7,
            doc_embeddings=np.array(emb).reshape(1, -1)  # use precomputed emb
        )]
    except:
        return []


In [8]:

# Run keyword extraction

tqdm.pandas()

print("Extracting YAKE keywords...")
df_all["yake_keywords"] = df_all["cleaned_text"].progress_apply(extract_yake)

print("Extracting KeyBERT keywords...")
df_all["keybert_keywords"] = [
    extract_keybert(text, emb)
    for text, emb in tqdm(zip(df_all["cleaned_text"], df_all["embedding"]), total=len(df_all))
]


Extracting YAKE keywords...


100%|██████████| 6000/6000 [01:09<00:00, 86.43it/s] 


Extracting KeyBERT keywords...


100%|██████████| 6000/6000 [09:35<00:00, 10.42it/s]


In [ ]:

# Merge keyword lists into one combined column

df_all["combined_keywords"] = df_all.apply(
    lambda row: list(dict.fromkeys(row["yake_keywords"] + row["keybert_keywords"])),
    axis=1
)


In [11]:
# ---------------------------
# 7️⃣ Save Phase 2 output
# ---------------------------
df_all.to_json("phase_2_keywords.json", orient="records", lines=True)
print("✅ Phase 2 complete: Keywords extracted and combined!")
print("Sample combined keywords:\n", df_all["combined_keywords"].iloc[0])


✅ Phase 2 complete: Keywords extracted and combined!
Sample combined keywords:
 ['asus', 'vivopc', 'window', 'desktop', 'time', 'faster', 'graphic', 'home', 'usb', 'gen', 'hardware', 'it049t', 'menu', 'save', 'online', 'iteration', 'percentage', 'filtering', 'forefront', 'blistering']
